<a href="https://colab.research.google.com/github/eniompw/TinyLM/blob/main/TorchMLP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
from datasets import load_dataset
import warnings, itertools
warnings.filterwarnings('ignore')

def load_tinystories(num_stories=200, context_size=4):
    """
    Fetches the TinyStories dataset and prepares it for a character-level language model.
    """
    # Fetch data
    dataset = load_dataset('karpathy/tinystories-gpt4-clean', split='train', streaming=True)
    text = ''.join(s['text'] for s in itertools.islice(dataset, num_stories))

    # Build vocabulary and tokenize
    vocab = sorted(set(text))                                       # ordered list of unique characters
    char_to_id = {c: i for i, c in enumerate(vocab)}                # dictionary mapping char to integer id
    encoded = [char_to_id[c] for c in text]                         # map entire text to integer sequence

    # If context_size=1, skip building windows to save RAM (training loop handles slicing).
    if context_size == 1:
        return [], [], vocab, encoded

    # Create sliding windows for inputs and targets using standard Python lists
    inputs = [encoded[i:i+context_size] for i in range(len(encoded)-context_size)] # sliding windows
    targets = encoded[context_size:]                                               # next char to predict

    return inputs, targets, vocab, encoded

In [4]:
import time, torch
import torch.nn as nn, torch.nn.functional as F
#from tinystories_dataset import load_tinystories

# Automatically create all tensors on GPU if available, removing manual device boilerplate
torch.set_default_device('cuda' if torch.cuda.is_available() else 'cpu')

# --- Data & Tokenization ---
input_ids, target_ids, idx_to_char, token_ids = load_tinystories(num_stories=200, context_size=4)# 4 is block_size (previous chars to predict next)
input_ids, target_ids = torch.tensor(input_ids), torch.tensor(target_ids)                        # convert to tensors

# --- Model ---
torch.manual_seed(42)                                                                             # seed helper for reproducibility
embedding = nn.Embedding(len(idx_to_char), 256)                                                  # token embedding lookup layer (256 is embed_dim)
mlp = nn.Sequential(
    nn.Linear(4 * 256, 150), nn.ReLU(),                                                          # maps context (4 * 256) to hidden_dim (150) & applies ReLU
    nn.Linear(150, len(idx_to_char))                                                             # maps hidden state (150) to logits (vocab length)
)
optimizer = torch.optim.SGD(list(embedding.parameters()) + list(mlp.parameters()), lr=0.5)       # optimizer replaces manual parameter updates

# --- Train ---
batch_size = 1024                                                                                 # number of samples per batch
start = time.time()                                                                               # track training duration
for step in range(2001):
    batch_idx = torch.randint(0, len(input_ids), (batch_size,))                                  # random batch indices (len(input_ids) is total examples)
    x = embedding(input_ids[batch_idx]).view(batch_size, -1)                                     # concatenate embeddings for the window
    loss = F.cross_entropy(mlp(x), target_ids[batch_idx])                                        # computes softmax and cross-entropy loss automatically
    optimizer.zero_grad(); loss.backward(); optimizer.step()                                      # zero grads, backprop, SGD update

    if step % 200 == 0:
        with torch.no_grad():                                                                     # disable tracking during evaluation
            pred_ids = mlp(embedding(input_ids).view(len(input_ids), -1)).argmax(1)              # full dataset forward & argmax
            print(f"Step {step:4d} | Loss: {loss:.4f} | Acc: {(pred_ids == target_ids).float().mean():.1%}")

print(f"Training time: {time.time() - start:.1f}s")

# --- Generate ---
@torch.no_grad()                                                                                  # disable autograd tracking during inference
def generate(num_chars=200, context_ids=list(token_ids[:4])):                                    # start with true initial context (4 chars)
    output_chars = [idx_to_char[i] for i in context_ids]                                         # decode initial context to string
    for _ in range(num_chars):
        next_token_probs = torch.softmax(mlp(embedding(torch.tensor([context_ids])).view(1, -1)), 1)  # fused forward pass
        next_token = torch.multinomial(next_token_probs, 1).item()                               # randomly sample from predicted distribution
        context_ids, output_chars = context_ids[1:] + [next_token], output_chars + [idx_to_char[next_token]]  # slide window forward and append string
    return ''.join(output_chars)

print(generate())

README.md:   0%|          | 0.00/3.30k [00:00<?, ?B/s]

Step    0 | Loss: 4.1034 | Acc: 21.4%
Step  200 | Loss: 1.6010 | Acc: 54.2%
Step  400 | Loss: 1.5188 | Acc: 57.6%
Step  600 | Loss: 1.3850 | Acc: 58.8%
Step  800 | Loss: 1.2806 | Acc: 60.0%
Step 1000 | Loss: 1.2455 | Acc: 60.4%
Step 1200 | Loss: 1.2568 | Acc: 61.3%
Step 1400 | Loss: 1.2753 | Acc: 60.8%
Step 1600 | Loss: 1.2401 | Acc: 61.9%
Step 1800 | Loss: 1.1818 | Acc: 61.3%
Step 2000 | Loss: 1.2581 | Acc: 62.2%
Training time: 4.4s
Once upon a time, to make".
The learr. They played would him in ort ar tried there said. "Mom, she saw while, the jo high small familbally any for and said.n Tim delp. He put her. After. "It's hort gragon
